In [7]:
import pandas as pd
import sys
sys.path.append('..')

from src.utils import create_player_base, safe_sum, update_rating

In [8]:
all_matches = pd.read_csv('../data/Setka.csv')

# Standard ELO, K=32 constant 

In [9]:
def process_matches(all_matches, player_base, metric_type="standard", K=32):
    
    results = []

    for match in all_matches.itertuples():
        player_A = match.Player1
        player_B = match.Player2
        r_old_A = player_base.loc[player_base['player'] == player_A, 'rating'].values[0]
        r_old_B = player_base.loc[player_base['player'] == player_B, 'rating'].values[0]
        
        actual_winner = "A" if match.HomeWinner == 1 else "B"
        
        home_points = safe_sum(match.P1_G1, match.P1_G2, match.P1_G3, match.P1_G4, match.P1_G5)
        away_points = safe_sum(match.P2_G1, match.P2_G2, match.P2_G3, match.P2_G4, match.P2_G5)
        
        match_data = {
            "sets_won": match.Sets_P1,
            "sets_lost": match.Sets_P2,
            "point_diff": abs(home_points - away_points)
        }

        r_new_A, r_new_B, E_A, E_B = update_rating(r_old_A, r_old_B, actual_winner, metric_type, match_data, K)

        
        # Record prediction BEFORE updating ratings
        results.append({
            'player1': player_A,
            'player2': player_B,
            'win_probability_p1': E_A,
            'actual_outcome': 1 if actual_winner == "A" else 0
        })
        
        # Now update ratings
        player_base.loc[player_base['player'] == player_A, 'rating'] = r_new_A
        player_base.loc[player_base['player'] == player_B, 'rating'] = r_new_B

    results_df = pd.DataFrame(results)
    results_df.to_csv(f'../results/{metric_type}_K{K}_results.csv', index=False)
    player_base.to_csv(f'../data/player_base_{metric_type}K{K}.csv', index=False)

In [4]:
player_base = create_player_base(all_matches)
process_matches(all_matches, player_base)

# Standard ELO, variable K

In [17]:
def process_matches_variable_k(all_matches, player_base, metric_type="standard",
                                K_new=64, K_veteran=32, veteran_threshold=15):

    results = []

    for match in all_matches.itertuples():
        player_A = match.Player1
        player_B = match.Player2

        r_old_A = player_base.loc[player_base['player'] == player_A, 'rating'].values[0]
        r_old_B = player_base.loc[player_base['player'] == player_B, 'rating'].values[0]

        n_A = player_base.loc[player_base['player'] == player_A, 'matches_played'].values[0]
        n_B = player_base.loc[player_base['player'] == player_B, 'matches_played'].values[0]

        K_A = K_new if n_A < veteran_threshold else K_veteran
        K_B = K_new if n_B < veteran_threshold else K_veteran

        actual_winner = "A" if match.HomeWinner == 1 else "B"
        home_points = safe_sum(match.P1_G1, match.P1_G2, match.P1_G3, match.P1_G4, match.P1_G5)
        away_points = safe_sum(match.P2_G1, match.P2_G2, match.P2_G3, match.P2_G4, match.P2_G5)

        # sets_won/sets_lost must reflect the ACTUAL winner, not P1 vs P2 --
        # Player1 isn't always the winner (see HomeWinner), so this has to
        # be conditional or the set-based multiplier silently breaks.
        if actual_winner == "A":
            sets_won, sets_lost = match.Sets_P1, match.Sets_P2
        else:
            sets_won, sets_lost = match.Sets_P2, match.Sets_P1

        match_data = {
            "sets_won": sets_won,
            "sets_lost": sets_lost,
            "point_diff": abs(home_points - away_points)
        }

        r_new_A, r_new_B, E_A, E_B = update_rating(
            r_old_A, r_old_B, actual_winner, metric_type, match_data, K_A, K_B
        )

        results.append({
            'player1': player_A,
            'player2': player_B,
            'win_probability_p1': E_A,
            'actual_outcome': 1 if actual_winner == "A" else 0,
            'K_A': K_A,
            'K_B': K_B,
        })

        player_base.loc[player_base['player'] == player_A, 'rating'] = r_new_A
        player_base.loc[player_base['player'] == player_B, 'rating'] = r_new_B
        player_base.loc[player_base['player'] == player_A, 'matches_played'] += 1
        player_base.loc[player_base['player'] == player_B, 'matches_played'] += 1

    results_df = pd.DataFrame(results)
    tag = f'{metric_type}_Kvar-new{K_new}-vet{K_veteran}-t{veteran_threshold}'
    results_df.to_csv(f'../results/{tag}_results.csv', index=False)
    player_base.to_csv(f'../data/player_base_{tag}.csv', index=False)

    return results_df, player_base

In [16]:
player_base = create_player_base(all_matches)
process_matches_variable_k(all_matches, player_base)

(           player1        player2  win_probability_p1  actual_outcome  K_A  \
 0     Hiblbauer J.     Kolacek I.              0.5000               1   64   
 1      Kleprlik K.    Pavliska M.              0.5000               1   64   
 2        Pikous M.       Pycha P.              0.5000               1   64   
 3       Schanel J.       Havel L.              0.5000               0   64   
 4         Pesek K.     Svoboda V.              0.5000               1   64   
 ...            ...            ...                 ...             ...  ...   
 4267       Tuma D.      Regner M.              0.4819               0   32   
 4268    Cernoch J.  Manhal Jr. J.              0.4710               1   32   
 4269   Havlicek V.     Hampejs V.              0.2941               1   32   
 4270   Janousek L.       Sudek P.              0.2735               1   32   
 4271      Kanok J.        Zika M.              0.3910               0   32   
 
       K_B  
 0      64  
 1      64  
 2      64 